This file has only one job that produce data for bronze laya. This data won't go to the table directly. We will save save in landing folder.
```text
02_generate_faker_data

Faker
  │
  ├── drivers
  ├── rides
  └── payments
       ↓
     JSON
       ↓
Volume/landing/

landing/
│
├── drivers/
│   ├── drivers_001.json
│   ├── drivers_002.json
│   └── ...
│
├── rides/
│   ├── rides_001.json
│   └── ...
│
└── payments/
    ├── payments_001.json
    └── ...
```

In [0]:
%pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 12.5 MB/s  0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

Faker is producing data continuously/in batches, while Spark Structured Streaming processes the arriving data as a stream

In [0]:
from faker import Faker
from datetime import datetime, timezone
import random
import json
import time

fake = Faker()

In [0]:
BASE_PATH = "/Volumes/ride_lakehouse_dev/bronze/streaming_files"

DRIVERS_PATH = f"{BASE_PATH}/landing/drivers"
RIDES_PATH = f"{BASE_PATH}/landing/rides"
PAYMENTS_PATH = f"{BASE_PATH}/landing/payments"

In [0]:
dbutils.widgets.text(
    "driver_count",
    "5"
)

dbutils.widgets.text(
    "ride_count",
    "20"
)

In [0]:
driver_count = int(
    dbutils.widgets.get("driver_count")
)

ride_count = int(
    dbutils.widgets.get("ride_count")
)

# Generate drivers
```text
drivers

id
name
car_number
experience
rating
generated_at

DRIVERS
id
 │
 │ 1:N
 ▼
RIDES
ride_id
 │
 │ 1:1
 ▼
PAYMENTS
```
There is no phone attritbe in Drivers data because I will test AutoLoader schema evoluation feature.

In [0]:
def generate_drivers(batch_id, count=10):
    drivers = []

    for i in range(count):

        driver = {
            "id": f"D-{batch_id}-{i:03d}", # Batch id makes them uniqe
            "name": fake.name(),
            "car_number": fake.bothify(text="??-####").upper(), # ? is letter # is digit
            "experience": random.randint(1, 20),
            "rating": round(random.uniform(1.0, 5.0), 2),
            "generated_at": datetime.now(timezone.utc).isoformat()
        }

        drivers.append(driver)

    return drivers

# Generate Rides
```Text
rides

ride_id
driver_id
customer_id
pickup_city
dropoff_city
distance_km
fare_amount
ride_timestamp
generated_at
```

In [0]:
def generate_rides(batch_id, drivers, count=50):

    rides = []

    driver_ids = [driver["id"] for driver in drivers]
    """
    driver_ids= []
    for driver in drivers:
        driver_ids.append(driver("id"))  
    this for loop is slightly slower than first expration because:
    1- Standard Python (CPython) optimizes list comprehensions to run closer to the underlying C code
    2- No .append() function overhead
    """
    for i in range(count):

        distance_km = round(random.uniform(1.0, 30.0), 2)

        fare_amount = round(
            5 + distance_km * random.uniform(1.5, 3.0),
            2
        )

        ride = {
            "ride_id": f"R-{batch_id}-{i:04d}",
            "driver_id": random.choice(driver_ids),
            "customer_id": fake.uuid4(),
            "pickup_city": fake.city(),
            "dropoff_city": fake.city(),
            "distance_km": distance_km,
            "fare_amount": fare_amount,
            "ride_timestamp": datetime.now(timezone.utc).isoformat(),
            "generated_at": datetime.now(timezone.utc).isoformat()
        }
        rides.append(ride)

    return rides


# Generate Payments
```text
payments

payment_id
ride_id
amount
payment_method
payment_timestamp
generated_at
```

In [0]:
def generate_payments(batch_id, rides):

    payment_methods = [
        "card",
        "cash",
        "wallet"
    ]

    payments = []

    for i, ride in enumerate(rides):
        # enumerate: automatically keeping track of the current item's index.
        payment = {
            "payment_id": f"P-{batch_id}-{i:04d}",
            "ride_id": ride["ride_id"],
            "amount": ride["fare_amount"],
            "payment_method": random.choice(payment_methods),
            "payment_timestamp": datetime.now(timezone.utc).isoformat(),
            "generated_at": datetime.now(timezone.utc).isoformat()
        }

        payments.append(payment)

    return payments

# Creating Batch
Create all entities in one batch
```text
generate_batch()

        │
        ▼
    batch_id
        │
        ├───────────────┐
        ▼               │
     drivers            │
        │               │
        ▼               │
      rides             │
        │               │
        ▼               │
    payments            │
                        │
All belong to same batch
```

In [0]:
def generate_batch(
    driver_count=10,
    ride_count=50
):

    batch_id = datetime.now(timezone.utc).strftime(
        "%Y%m%d%H%M%S%f"
    )

    drivers = generate_drivers(
        batch_id=batch_id,
        count=driver_count
    )

    rides = generate_rides(
        batch_id=batch_id,
        drivers=drivers,
        count=ride_count
    )

    payments = generate_payments(
        batch_id=batch_id,
        rides=rides
    )

    return batch_id, drivers, rides, payments

## Test


In [0]:
# batch_id, drivers, rides, payments = generate_batch(
#     driver_count=5,
#     ride_count=10
# )
#I am puttin tessting cells into comment because aoutomaic jobs runs all cells

In [0]:
#drivers[:2]

[{'id': 'D-20260929112626334496-000',
  'name': 'Caroline Phillips',
  'car_number': 'NU-1128',
  'experience': 15,
  'rating': 3.62,
  'generated_at': '2026-09-29T11:26:26.334871+00:00'},
 {'id': 'D-20260929112626334496-001',
  'name': 'Julia Garrison',
  'car_number': 'SD-8666',
  'experience': 2,
  'rating': 2.48,
  'generated_at': '2026-09-29T11:26:26.335042+00:00'}]

In [0]:
#rides[:2]

[{'ride_id': 'R-20260929112626334496-0000',
  'driver_id': 'D-20260929112626334496-004',
  'customer_id': 'c335de90-3fc2-4df9-8c3c-57efd6b1984f',
  'pickup_city': 'Moorefort',
  'dropoff_city': 'Judyport',
  'distance_km': 19.19,
  'fare_amount': 38.63,
  'ride_timestamp': '2026-09-29T11:26:26.335738+00:00',
  'generated_at': '2026-09-29T11:26:26.335742+00:00'},
 {'ride_id': 'R-20260929112626334496-0001',
  'driver_id': 'D-20260929112626334496-004',
  'customer_id': '26ae1c42-fd2f-4abc-9da5-2ea653273297',
  'pickup_city': 'West Julie',
  'dropoff_city': 'Alexanderborough',
  'distance_km': 11.68,
  'fare_amount': 29.6,
  'ride_timestamp': '2026-09-29T11:26:26.335912+00:00',
  'generated_at': '2026-09-29T11:26:26.335916+00:00'}]

In [0]:
#payments[:2]

[{'payment_id': 'P-20260929112626334496-0000',
  'ride_id': 'R-20260929112626334496-0000',
  'amount': 38.63,
  'payment_method': 'card',
  'payment_timestamp': '2026-09-29T11:26:26.337148+00:00',
  'generated_at': '2026-09-29T11:26:26.337150+00:00'},
 {'payment_id': 'P-20260929112626334496-0001',
  'ride_id': 'R-20260929112626334496-0001',
  'amount': 29.6,
  'payment_method': 'card',
  'payment_timestamp': '2026-09-29T11:26:26.337154+00:00',
  'generated_at': '2026-09-29T11:26:26.337156+00:00'}]

# Faker and Source System
At this moment all our data is in lists. Our goal is saving data in JSON format in volume (that in bronze layer). Then Spark will read our data from there.

## Creating JSON file (as a batch)

In [0]:
def write_json_batch(
    records,
    directory,
    prefix,
    batch_id
):

    file_path = (
        f"{directory}/"
        f"{prefix}_batch_{batch_id}.json"
    )

    json_content = "\n".join(
        json.dumps(record)
        for record in records
    )
    """
    json_content is so important because we save each json line in this file instead saving them as a json list. Thanks to this, we can read the file line by line and process each line as a json object (spitable files from spark therical part), instead reading whole list.
    """

    dbutils.fs.put(
        file_path,
        json_content,
        overwrite=False
    )

    return file_path

# Write one complete batch

```text
generate_and_write_batch()
          │
          ▼
     generate_batch()
          │
       ┌──┼──┐
       ▼  ▼  ▼
       D  R  P
       │  │  │
       ▼  ▼  ▼
     JSON JSON JSON
       │   │   │
       └───┼───┘
           ▼
        Volume
```

In [0]:
def generate_and_write_batch(
    driver_count=10,
    ride_count=50
):

    batch_id, drivers, rides, payments = generate_batch(
        driver_count=driver_count,
        ride_count=ride_count
    )

    driver_file = write_json_batch(
        records=drivers,
        directory=DRIVERS_PATH,
        prefix="drivers",
        batch_id=batch_id
    )

    ride_file = write_json_batch(
        records=rides,
        directory=RIDES_PATH,
        prefix="rides",
        batch_id=batch_id
    )

    payment_file = write_json_batch(
        records=payments,
        directory=PAYMENTS_PATH,
        prefix="payments",
        batch_id=batch_id
    )

    print(f"Batch ID: {batch_id}")

    print(f"Drivers:  {len(drivers)}")
    print(f"Rides:    {len(rides)}")
    print(f"Payments: {len(payments)}")

    print("\nFiles created:")

    print(driver_file)
    print(ride_file)
    print(payment_file)

    return batch_id

In [0]:
generate_and_write_batch(
    driver_count=driver_count,
    ride_count=ride_count
)

Wrote 1721 bytes.
Wrote 17728 bytes.
Wrote 12020 bytes.
Batch ID: 20260929114544667600
Drivers:  10
Rides:    50
Payments: 50

Files created:
/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114544667600.json
/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114544667600.json
/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114544667600.json


'20260929114544667600'

At this moment, our data met with bronze layer landing enviorment(volume). Volumes are suitable for non-tabular data and databricks can scale this enviorment automaticaly which makes spark/databricks to read or gateher data from there easly

## Create more data

In [0]:
# for _ in range(3):

#     generate_and_write_batch(
#         driver_count=10,
#         ride_count=50
#     )

#     time.sleep(2)

Wrote 1712 bytes.
Wrote 17610 bytes.
Wrote 12028 bytes.
Batch ID: 20260929114909008713
Drivers:  10
Rides:    50
Payments: 50

Files created:
/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114909008713.json
/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114909008713.json
/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114909008713.json
Wrote 1737 bytes.
Wrote 17702 bytes.
Wrote 12029 bytes.
Batch ID: 20260929114913448298
Drivers:  10
Rides:    50
Payments: 50

Files created:
/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json
/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/rides/rides_batch_20260929114913448298.json
/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/payments/payments_batch_20260929114913448298.json
Wrote 1718 bytes.
Wrote 17702 bytes.
Wrote 12036 bytes.
Batch ID: 202609291149

# But is this really streaming?
Each event (actions like creating driver, ride or payment) are gathering and create a small file. Then they are send as a batch to our bronze layer volume. Until autoloader discover them and spark structured streaming triggers its next micro-batch to read it. Clinet can not see the result. Based on the confiurtion, ıt might take 5 minutes to seconds. That's called "near-real time"

Also application might be supported by high-speed distributed queue like Apache Kafka...